# RSNA Knee — Submission Only Notebook

This notebook uses the trained `best_study_dinov2.pth` checkpoint from the existing project.
It does **not** train the model.

Before running:
- Add the RSNA competition data to the notebook.
- Put/upload `best_study_dinov2.pth` into `/kaggle/working/`.


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import timm
import pydicom

from PIL import Image
from torch.utils.data import Dataset, DataLoader

DEVICE = torch.device("cpu")

DATA_ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
CHECKPOINT_PATH = "/kaggle/input/datasets/rakhikrishnaau/rsna-knee-dinov2-checkpoint/best_study_dinov2.pth"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

PLANES = ["Axial", "Coronal", "Sagittal"]

LABEL_NAMES = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture"
]

print("Device:", DEVICE)
print("Data root exists:", os.path.exists(DATA_ROOT))
print("Checkpoint exists:", os.path.exists(CHECKPOINT_PATH))


In [ ]:
# Load official test study IDs

test_csv_path = os.path.join(DATA_ROOT, "test.csv")
test_df = pd.read_csv(test_csv_path)

test_study_ids = test_df["StudyInstanceUID"].astype(str).tolist()

print("Test studies:", len(test_study_ids))
print(test_df.head())


In [ ]:
# Build test-series lookup and select one series per plane.
# This follows the existing notebook's test-series selection logic.

TEST_SERIES_CSV = os.path.join(DATA_ROOT, "test_series.csv")
test_series_df = pd.read_csv(TEST_SERIES_CSV)

from collections import defaultdict

test_series_lookup = defaultdict(lambda: defaultdict(list))

for _, row in test_series_df.iterrows():
    study_id = str(row["StudyInstanceUID"])
    series_id = str(row["SeriesInstanceUID"])
    plane = str(row["Anatomical_Plane"])

    test_series_lookup[study_id][plane].append({
        "series_id": series_id,
        "fluid_sensitive": int(row["Fluid_Sensitive"]),
        "fat_suppression": int(row["Fat_Suppression"])
    })

test_selected_series = {}

for study_id in test_study_ids:
    test_selected_series[study_id] = {}

    for plane in PLANES:
        candidates = test_series_lookup[study_id][plane]

        preferred = [
            x for x in candidates
            if x["fluid_sensitive"] == 1
            and x["fat_suppression"] == 1
        ]

        if len(preferred) == 0:
            preferred = [
                x for x in candidates
                if x["fluid_sensitive"] == 1
            ]

        if len(preferred) == 0:
            preferred = candidates

        test_selected_series[study_id][plane] = preferred[0]["series_id"]

print("Selected test series for", len(test_selected_series), "studies.")


In [ ]:
# Exact test image loader from the existing notebook

def load_test_series_25d_336(
    study_id,
    plane,
    num_slices=3,
    image_size=336
):
    series_id = test_selected_series[study_id][plane]

    series_dir = os.path.join(
        DATA_ROOT,
        "test_series",
        study_id,
        series_id
    )

    dicom_files = [
        os.path.join(series_dir, f)
        for f in os.listdir(series_dir)
        if f.lower().endswith(".dcm")
    ]

    if len(dicom_files) == 0:
        raise RuntimeError(f"No DICOM files found:\n{series_dir}")

    slices = []

    for path in dicom_files:
        try:
            ds = pydicom.dcmread(path, force=True)

            if not hasattr(ds, "PixelData"):
                continue

            ipp = getattr(ds, "ImagePositionPatient", None)
            iop = getattr(ds, "ImageOrientationPatient", None)

            if ipp is None or iop is None:
                continue

            ipp = np.asarray(ipp, dtype=np.float32)
            iop = np.asarray(iop, dtype=np.float32)

            row = iop[:3]
            col = iop[3:]
            normal = np.cross(row, col)
            position = float(np.dot(ipp, normal))

            image = ds.pixel_array.astype(np.float32)

            slope = float(getattr(ds, "RescaleSlope", 1.0))
            intercept = float(getattr(ds, "RescaleIntercept", 0.0))
            image = image * slope + intercept

            slices.append({"position": position, "image": image})

        except Exception:
            continue

    if len(slices) == 0:
        raise RuntimeError(f"No readable DICOM slices:\n{series_dir}")

    slices.sort(key=lambda x: x["position"])

    center = len(slices) // 2
    start = max(0, center - num_slices // 2)

    indices = list(
        range(start, min(start + num_slices, len(slices)))
    )

    while len(indices) < num_slices:
        indices.append(indices[-1])

    output = []

    for idx in indices:
        image = slices[idx]["image"]

        p1, p99 = np.percentile(image, [1, 99])

        if p99 > p1:
            image = np.clip(
                (image - p1) / (p99 - p1),
                0,
                1
            )
        else:
            image = np.zeros_like(image)

        image = (image * 255).astype(np.uint8)

        image = Image.fromarray(image).resize(
            (image_size, image_size),
            Image.Resampling.BILINEAR
        )

        image = np.asarray(image, dtype=np.float32) / 255.0
        image = np.stack([image, image, image], axis=0)

        output.append(image)

    return np.stack(output, axis=0)


In [ ]:
# Test dataset

class KneeTestDataset(Dataset):

    def __init__(self, study_ids, image_size=336):
        self.study_ids = list(study_ids)
        self.image_size = image_size

    def __len__(self):
        return len(self.study_ids)

    def __getitem__(self, idx):
        study_id = self.study_ids[idx]

        plane_images = []

        for plane in PLANES:
            images = load_test_series_25d_336(
                study_id,
                plane,
                num_slices=3,
                image_size=self.image_size
            )
            plane_images.append(images)

        image = np.stack(plane_images, axis=0)
        image = torch.from_numpy(image).float()

        return {
            "study_id": study_id,
            "image": image
        }

test_dataset = KneeTestDataset(test_study_ids, image_size=336)

test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("Test studies:", len(test_dataset))
print("Test batches:", len(test_loader))


In [ ]:
# Exact trained study-level DINOv2 model

backbone = timm.create_model(
    "vit_small_patch14_dinov2.lvd142m",
    pretrained=False,
    num_classes=0,
    img_size=336
)

class StudyDINOv2Classifier(nn.Module):

    def __init__(self, backbone):
        super().__init__()

        self.backbone = backbone
        self.feature_dim = 384

        self.slice_attention = nn.Sequential(
            nn.Linear(384, 128),
            nn.GELU(),
            nn.Linear(128, 1)
        )

        self.plane_attention = nn.Sequential(
            nn.Linear(384, 128),
            nn.GELU(),
            nn.Linear(128, 1)
        )

        self.head = nn.Sequential(
            nn.LayerNorm(384),
            nn.Linear(384, 256),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(256, 12)
        )

    def forward(self, x):
        B = x.shape[0]

        x = x.reshape(B * 9, 3, 336, 336)
        features = self.backbone(x)
        features = features.reshape(B, 3, 3, 384)

        slice_scores = self.slice_attention(features)
        slice_weights = torch.softmax(slice_scores, dim=2)

        plane_features = (
            features * slice_weights
        ).sum(dim=2)

        plane_scores = self.plane_attention(plane_features)
        plane_weights = torch.softmax(plane_scores, dim=1)

        fused = (
            plane_features * plane_weights
        ).sum(dim=1)

        return self.head(fused)


study_model = StudyDINOv2Classifier(backbone).to(DEVICE)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

state_dict = checkpoint["model_state_dict"]

clean_state_dict = {}
for key, value in state_dict.items():
    if key.startswith("module."):
        key = key[len("module."):]
    clean_state_dict[key] = value

missing, unexpected = study_model.load_state_dict(
    clean_state_dict,
    strict=True
)

study_model.eval()

print("CHECKPOINT LOADED SUCCESSFULLY")
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)
print("Epoch:", checkpoint["epoch"])
print("Best loss:", checkpoint["best_loss"])


In [ ]:
# Run test inference

test_predictions = []
test_ids = []

with torch.no_grad():
    for batch_idx, batch in enumerate(test_loader):

        images = batch["image"].to(
            DEVICE,
            non_blocking=True
        )

        logits = study_model(images)

        probabilities = torch.sigmoid(logits)

        test_predictions.append(
            probabilities.detach().cpu().numpy()
        )

        test_ids.extend(batch["study_id"])

        print(
            f"Processed {batch_idx + 1}/{len(test_loader)}"
        )

test_predictions = np.concatenate(
    test_predictions,
    axis=0
)

print("Prediction shape:", test_predictions.shape)
print(
    "Prediction range:",
    float(test_predictions.min()),
    "to",
    float(test_predictions.max())
)


In [ ]:
# Create and validate submission.csv

submission = pd.DataFrame(
    test_predictions,
    columns=LABEL_NAMES
)

submission.insert(
    0,
    "StudyInstanceUID",
    test_ids
)

submission.to_csv(
    SUBMISSION_PATH,
    index=False
)

check = pd.read_csv(SUBMISSION_PATH)

expected_columns = ["StudyInstanceUID"] + LABEL_NAMES

print("Shape:", check.shape)
print("Missing values:", check.isnull().sum().sum())
print(
    "Values outside [0,1]:",
    ((check[LABEL_NAMES] < 0) | (check[LABEL_NAMES] > 1)).sum().sum()
)
print(
    "Duplicate StudyInstanceUIDs:",
    check["StudyInstanceUID"].duplicated().sum()
)
print("Column order correct:", list(check.columns) == expected_columns)
print("File:", SUBMISSION_PATH)

assert list(check.columns) == expected_columns
assert check["StudyInstanceUID"].duplicated().sum() == 0
assert check[LABEL_NAMES].notna().all().all()
assert np.isfinite(check[LABEL_NAMES].values).all()
assert ((check[LABEL_NAMES] >= 0) & (check[LABEL_NAMES] <= 1)).all().all()

print("\n✓ submission.csv READY")


In [ ]:
import os

print(os.listdir("/kaggle/input"))

In [ ]:
import os

print(os.listdir("/kaggle/input/datasets"))

In [ ]:
import os

print(os.listdir("/kaggle/input/datasets/rakhikrishnaau"))

In [ ]:
import os

print(os.listdir(
    "/kaggle/input/datasets/rakhikrishnaau/rsna-knee-dinov2-checkpoint"
))